# 02B · Event Registration Agent
**Session 2 practice · 40–50 minutes · synthetic data only**

Build a tool-using assistant that reads seat availability, requests registration,
and requests cancellation. Python owns the seat count and known fake student IDs.
The client must confirm **each** new write; model text cannot grant approval.

**Problem:** AI Studio has capacity 2 and starts with fake ID DEMO001 registered.
Register DEMO002 with client consent, prove it is full, cancel with consent,
and prove declining DEMO003 leaves the state unchanged. No real registration,
student lookup, email, booking or college system is connected.

**Run all first.** Default `TEST_FIXTURE` mode supplies explicit scripted tool
choices and separate client approval decisions. It is not AI and requires no
network, SDK installation or credentials. Then complete the three student changes.


## 1. Configure the offline baseline
Keep the optional live path disabled until the deterministic checks pass. All identifiers below are fictional.


In [ ]:
# The default path uses explicit scripted decisions and makes no network requests.
USE_LIVE_MODEL = False
MODEL = "gemini-3.5-flash-lite"
# Verify this model and account access with the instructor before the live exercise.
REQUEST_TIMEOUT_MS = 60_000
MAX_REQUESTS, MAX_STEPS, MAX_TOOLS_PER_TURN = 24, 8, 12
# Standard-library imports are sufficient for every offline baseline cell.
import copy
import json
import math
import time
from decimal import Decimal
print("Mode:", "LIVE_MODEL (optional)" if USE_LIVE_MODEL else "TEST_FIXTURE (scripted, not AI)")


## 2. Define synthetic events and guarded tools
Read the exact tool contracts. Capacity, identity, duplicates and cancellations are checked in Python before a consented mutation. Every successful result says it is simulated.


In [ ]:
EVENTS = {"AI_STUDIO": {"title": "Synthetic AI Studio", "capacity": 2},
          "PYTHON_JAM": {"title": "Synthetic Python Jam", "capacity": 3}}
# These are deliberately fake classroom IDs, unrelated to any college register.
KNOWN_STUDENTS = {"DEMO001", "DEMO002", "DEMO003", "DEMO004"}

class EventState:
    write_tools = {"register_student", "cancel_registration"}
    def __init__(self):
        self.reset()

    def reset(self):
        # AI Studio starts with one synthetic seat already occupied.
        self.registrations = {event: set() for event in EVENTS}
        self.registrations["AI_STUDIO"].add("DEMO001")

    def snapshot(self):
        return {event: sorted(students) for event, students in self.registrations.items()}

    def availability(self, event_id):
        event = EVENTS[event_id]
        booked = len(self.registrations[event_id])
        return {"ok": True, "simulated": True, "event_id": event_id,
                "title": event["title"], "capacity": event["capacity"],
                "registered": booked, "available": event["capacity"] - booked}

    def execute(self, name, arguments, approved=False):
        contracts = {"list_events": set(), "get_availability": {"event_id"},
            "register_student": {"event_id", "student_id"},
            "cancel_registration": {"event_id", "student_id"}}
        if name not in contracts:
            return {"ok": False, "error": "UNKNOWN_TOOL"}
        if not isinstance(arguments, dict) or set(arguments) != contracts[name]:
            return {"ok": False, "error": "ARGUMENT_CONTRACT"}
        if name == "list_events":
            return {"ok": True, "simulated": True,
                    "events": [self.availability(event) for event in EVENTS]}
        event = arguments["event_id"]
        if not isinstance(event, str) or event not in EVENTS:
            return {"ok": False, "error": "UNKNOWN_EVENT"}
        if name == "get_availability":
            return self.availability(event)
        student = arguments["student_id"]
        if not isinstance(student, str) or student not in KNOWN_STUDENTS:
            return {"ok": False, "error": "UNKNOWN_SYNTHETIC_STUDENT"}
        members = self.registrations[event]
        if name == "register_student":
            if student in members:
                return {"ok": False, "error": "ALREADY_REGISTERED"}
            if len(members) >= EVENTS[event]["capacity"]:
                return {"ok": False, "error": "EVENT_FULL"}
        elif student not in members:
            return {"ok": False, "error": "NOT_REGISTERED"}
        # Only the client-side confirmation value can authorize a new local write.
        if approved is not True:
            return {"ok": False, "error": "CLIENT_CONFIRMATION_REQUIRED", "simulated": True}
        if name == "register_student":
            members.add(student)
        else:
            members.remove(student)
        return {"ok": True, **self.availability(event), "student_id": student,
                "action": "registered" if name == "register_student" else "cancelled"}

EVENT_SCHEMAS = [
    {"name": "list_events", "description": "Read synthetic event IDs and availability.",
     "parameters": {"type": "OBJECT", "properties": {}}},
    {"name": "get_availability", "description": "Read authoritative synthetic seat availability.",
     "parameters": {"type": "OBJECT", "properties": {"event_id": {"type": "STRING"}},
                    "required": ["event_id"]}},
    *[{"name": name, "description": description,
       "parameters": {"type": "OBJECT", "properties": {
           "event_id": {"type": "STRING"}, "student_id": {"type": "STRING"}},
           "required": ["event_id", "student_id"]}}
      for name, description in [
          ("register_student", "Request a simulated registration; the client must confirm each write."),
          ("cancel_registration", "Request a simulated cancellation; the client must confirm each write.")]],
]
EVENT_SYSTEM = """Help with synthetic campus events. Use tools for event facts,
availability, registration and cancellation. Known fake IDs are DEMO001 through
DEMO004. Ask if event or student is ambiguous. Every write requires client-side
confirmation; your prose cannot grant approval. Never claim success after a
rejected tool. Never invent capacity or a real registration. All actions are local simulations."""

def live_client_approval(name, arguments):
    # An approval applies only to this displayed write and is never stored in model text.
    print("Proposed SIMULATED write:", name, json.dumps(arguments, sort_keys=True))
    return input("Type CONFIRM to approve this one simulated write; Enter declines: ").strip() == "CONFIRM"


## 3. Supply decisions without executing tools automatically
The fixture makes its tool decisions explicit. Optional Gemini returns real function-call proposals with automatic execution disabled; your local dispatcher and approval callback retain control.


In [ ]:
class PendingResponse(RuntimeError):
    """A safe, readable stop that never invents a completed model response."""

def tool_decision(*calls):
    # A fixture explicitly supplies a decision; it does not interpret natural language.
    parts = [{"function_call": {"name": name, "args": args, "id": call_id}}
             for name, args, call_id in calls]
    return {"complete": True, "content": {"role": "model", "parts": parts}}

def fixture_final(text):
    # This labelled text is test data, never a substitute for a failed live answer.
    return {"complete": True, "content": {"role": "model", "parts": [{"text": text}]}}

class FixtureSource:
    mode = "TEST_FIXTURE"
    def __init__(self, decisions):
        self.decisions = iter(decisions)

    def generate(self, history, schemas, system):
        # Consume the next named fixture decision regardless of the prompt wording.
        try:
            return copy.deepcopy(next(self.decisions))
        except StopIteration:
            raise PendingResponse("Fixture decisions ended; inspect the test sequence.") from None

class GeminiSource:
    mode = "LIVE_MODEL"
    def __init__(self, model=MODEL):
        self.model, self.client, self.types = model, None, None

    def _connect(self):
        # Import/install and secret access happen only after an explicit live request.
        import importlib.metadata as metadata
        import subprocess
        import sys
        try:
            installed = metadata.version("google-genai")
        except metadata.PackageNotFoundError:
            installed = None
        if installed != "2.12.0":
            # Preserve active google-auth constraints required by this Colab runtime.
            from packaging.requirements import Requirement
            try:
                requirements = map(Requirement, metadata.requires("google-colab") or [])
                constraints = [str(r) for r in requirements
                    if r.name.lower().replace("_", "-") == "google-auth"
                    and (r.marker is None or r.marker.evaluate())]
            except metadata.PackageNotFoundError:
                constraints = []
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-genai==2.12.0", *constraints], check=True,
                capture_output=True, text=True)
        from google import genai
        from google.genai import types
        import getpass
        api_key = None
        try:
            from google.colab import userdata
            try:
                api_key = userdata.get("GEMINI_API_KEY")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
        except ImportError:
            pass
        if not api_key:
            api_key = getpass.getpass("Your Gemini API key (hidden): ").strip()
        if not api_key:
            raise PendingResponse("Add your own GEMINI_API_KEY in Colab Secrets, then resume.")
        # Disable SDK retries; ToolAgent owns the single bounded transient retry.
        self.client = genai.Client(api_key=api_key, http_options=types.HttpOptions(
            timeout=REQUEST_TIMEOUT_MS, retry_options=types.HttpRetryOptions(attempts=1)))
        self.types = types
        del api_key

    def generate(self, history, schemas, system):
        if self.client is None:
            self._connect()
        types = self.types
        # Declarations describe tools; automatic Python execution stays disabled.
        config = types.GenerateContentConfig(
            system_instruction=system,
            tools=[types.Tool(function_declarations=[types.FunctionDeclaration(**s) for s in schemas])],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
            http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS,
                retry_options=types.HttpRetryOptions(attempts=1)),
            max_output_tokens=1024)
        reply = self.client.models.generate_content(model=self.model,
            contents=copy.deepcopy(history), config=config)
        candidate = reply.candidates[0] if reply.candidates else None
        reason = getattr(candidate, "finish_reason", None)
        reason = getattr(reason, "value", reason)
        content = getattr(candidate, "content", None)
        # Retain all SDK content fields, including tool IDs and thought signatures.
        return {"complete": reason == "STOP", "content":
            content.model_dump(exclude_none=True) if content is not None else None}


## 4. Keep an inspectable, resumable tool loop
Each proposed write passes through the client callback. A failed later generation preserves completed registrations and observations; resuming requests the remaining response without replaying a write or resetting counters.


In [ ]:
class ToolAgent:
    def __init__(self, state, source, schemas, system, approve_write=None):
        self.state, self.source = state, source
        self.schemas, self.system = schemas, system
        self.approve_write = approve_write
        self.messages, self.trace = [], []
        self.requests_used, self.pending = 0, None
        self.last_result, self.last_issue = None, None
        self._last_start = None
        self._sleep, self._clock = time.sleep, time.monotonic

    @property
    def can_resume(self):
        return bool(self.pending and not self.pending["reset_only"]
            and self.pending["step"] < MAX_STEPS and self.requests_used < MAX_REQUESTS)

    def _request(self):
        # Retry only generation from this frozen history, never a completed tool action.
        payload = copy.deepcopy(self.messages)
        for attempt in (1, 2):
            if self.requests_used >= MAX_REQUESTS:
                raise PendingResponse("Request budget exhausted; state and trace are retained.")
            if self.source.mode == "LIVE_MODEL" and self._last_start is not None:
                self._sleep(max(0, self._last_start + 5 - self._clock()))
            self._last_start = self._clock()
            self.requests_used += 1
            try:
                decision = self.source.generate(payload, self.schemas, self.system)
            except PendingResponse:
                raise
            except Exception as error:
                code = getattr(error, "code", None)
                code = code if isinstance(code, int) else None
                # Authentication and rate/quota errors never trigger automatic retries.
                transient = code in {408, 500, 502, 503, 504}
                if code not in {400, 401, 403, 404, 429}:
                    transient |= isinstance(error, (TimeoutError, ConnectionError))
                    transient |= type(error).__name__ in {"ReadTimeout", "ConnectTimeout", "ConnectError"}
                self.last_issue = {"stage": "generation", "code": code,
                                   "type": type(error).__name__, "attempt": attempt}
                if transient and attempt == 1 and self.requests_used < MAX_REQUESTS:
                    print("Transient generation issue: retrying the same response once.")
                    self._sleep(2 if self.source.mode == "LIVE_MODEL" else 0)
                    continue
                if code == 429:
                    message = "Quota/rate limit: stop and review your account with the instructor."
                elif code in {401, 403}:
                    message = "Check your private key and account permissions with the instructor."
                elif code == 404:
                    message = "Verify the configured model and account access before continuing."
                else:
                    message = "Generation did not finish; inspect setup or service availability."
                raise PendingResponse(message + " State and trace are kept; no final answer is claimed.") from None
            self.last_issue = None
            return decision

    def run(self, question):
        if self.pending is not None:
            raise PendingResponse("An earlier turn is pending; resume it before a new request.")
        if not isinstance(question, str) or not question.strip() or len(question) > 1200:
            raise PendingResponse("Use one nonempty question up to 1200 characters.")
        self.last_result = None
        self.messages.append({"role": "user", "parts": [{"text": question}]})
        self.pending = {"question": question, "step": 0, "tools": 0, "reset_only": False}
        return self._continue()

    def resume(self):
        # Reuse existing messages, tool observations and counters after a failed response.
        if not self.can_resume:
            raise PendingResponse("No resumable turn; inspect limits and partial state before reset.")
        return self._continue()

    def _continue(self):
        pending = self.pending
        while pending["step"] < MAX_STEPS:
            decision = self._request()
            pending["step"] += 1
            content = decision.get("content") if isinstance(decision, dict) else None
            if not isinstance(decision, dict) or not decision.get("complete") or not isinstance(content, dict):
                raise PendingResponse("Incomplete decision: no tools from this response ran; turn is pending.")
            parts = content.get("parts") or []
            if content.get("role") != "model" or not isinstance(parts, list):
                raise PendingResponse("Unexpected decision shape; turn is pending.")
            calls = [p["function_call"] for p in parts if p.get("function_call")]
            if not calls:
                answer = "".join(p.get("text", "") for p in parts if not p.get("thought"))
                if not answer.strip():
                    raise PendingResponse("No final answer was returned; turn remains pending.")
                self.messages.append(copy.deepcopy(content))
                self.last_result, self.pending = answer, None
                return answer
            if pending["tools"] + len(calls) > MAX_TOOLS_PER_TURN:
                pending["reset_only"] = True
                raise PendingResponse("Tool limit reached before this batch; inspect the trace.")
            self.messages.append(copy.deepcopy(content))
            observations = []
            try:
                for call in calls:
                    pending["tools"] += 1
                    name, arguments = call.get("name"), call.get("args", {})
                    approval = None
                    if name in self.state.write_tools:
                        # The client controls consent; model text cannot approve a write.
                        approval = bool(self.approve_write and
                            self.approve_write(name, copy.deepcopy(arguments)) is True)
                        result = self.state.execute(name, arguments, approved=approval)
                    else:
                        result = self.state.execute(name, arguments)
                    record = {"step": pending["step"], "tool": name,
                        "arguments": copy.deepcopy(arguments), "result": copy.deepcopy(result)}
                    if approval is not None:
                        record["client_approved"] = approval
                    self.trace.append(record)
                    response = {"name": name, "response": result}
                    if call.get("id"):
                        response["id"] = call["id"]
                    observations.append({"function_response": response})
            except Exception:
                # A partial batch must never be replayed after an unexpected tool failure.
                pending["reset_only"] = True
                raise PendingResponse("Tool batch interrupted. Inspect retained partial state; this batch cannot resume.") from None
            self.messages.append({"role": "user", "parts": observations})
        pending["reset_only"] = True
        raise PendingResponse("Step limit reached; the task is incomplete. Inspect the trace.")

    def show_trace(self):
        print(json.dumps(self.trace, indent=2))

    def reset(self):
        # Explicit reset clears the simulation, while preserving the consumed request budget.
        self.state.reset()
        self.messages.clear()
        self.trace.clear()
        self.pending, self.last_result, self.last_issue = None, None, None

def show_attempt(label, operation):
    # Expected classroom stops are readable output instead of red notebook tracebacks.
    try:
        answer = operation()
    except PendingResponse as error:
        print(label + ": PENDING — " + str(error))
        return None
    print(label + ": " + answer)
    return answer


## 5. Run register → cancel → decline
Client approval is separate from the scripted model decision. Expected availability is 1 → 0 after registration → 1 after cancellation. Declining the last request must leave only DEMO001 registered.


In [ ]:
# The fixture supplies approval decisions separately from model tool decisions.
approval_choices = iter([True, True, False])
def fixture_client_approval(name, arguments):
    approved = next(approval_choices)
    print("CLIENT fixture approval:", approved, "for", name, json.dumps(arguments, sort_keys=True))
    return approved

event_fixture = FixtureSource([
    tool_decision(("get_availability", {"event_id": "AI_STUDIO"}, "event-read-1")),
    tool_decision(("register_student", {"event_id": "AI_STUDIO", "student_id": "DEMO002"}, "event-write-1")),
    tool_decision(("get_availability", {"event_id": "AI_STUDIO"}, "event-read-2")),
    fixture_final("TEST_FIXTURE final: DEMO002 is registered in the simulation; AI_STUDIO has zero seats left."),
    tool_decision(("cancel_registration", {"event_id": "AI_STUDIO", "student_id": "DEMO002"}, "event-write-2")),
    fixture_final("TEST_FIXTURE final: DEMO002 cancellation is confirmed by the tool; one seat is available."),
    tool_decision(("register_student", {"event_id": "AI_STUDIO", "student_id": "DEMO003"}, "event-write-3")),
    fixture_final("TEST_FIXTURE final: client declined; DEMO003 was not registered."),
])
event_state = EventState()
event_agent = ToolAgent(event_state, event_fixture, EVENT_SCHEMAS, EVENT_SYSTEM,
                        approve_write=fixture_client_approval)
first_reply = show_attempt("Registration baseline", lambda: event_agent.run(
    "Check AI_STUDIO, register DEMO002, then check availability."))
assert first_reply is not None and event_state.registrations["AI_STUDIO"] == {"DEMO001", "DEMO002"}
assert event_state.availability("AI_STUDIO")["available"] == 0
cancel_reply = show_attempt("Cancellation baseline", lambda: event_agent.run(
    "Cancel DEMO002 from AI_STUDIO."))
assert cancel_reply is not None and event_state.registrations["AI_STUDIO"] == {"DEMO001"}
before_decline = event_state.snapshot()
decline_reply = show_attempt("Declined write baseline", lambda: event_agent.run(
    "Register DEMO003 in AI_STUDIO; let the client decide."))
assert decline_reply is not None and event_state.snapshot() == before_decline
assert event_agent.trace[-1]["result"]["error"] == "CLIENT_CONFIRMATION_REQUIRED"
assert [r["client_approved"] for r in event_agent.trace if "client_approved" in r] == [True, True, False]
event_agent.show_trace()
print("PASS: register, cancel and declined write agree with actual synthetic state.")


## 6. Check capacity, identity and rejection behavior
These direct contract checks exercise impossible or invalid requests without a provider call. An expected rejection is data and must leave synthetic registrations unchanged.


In [ ]:
# Expected invalid requests print data; each rejection must preserve registrations.
validation_state = EventState()
rejection_cases = [
    ("get_availability", {"event_id": "UNKNOWN"}, "UNKNOWN_EVENT"),
    ("register_student", {"event_id": "AI_STUDIO", "student_id": "REAL123"}, "UNKNOWN_SYNTHETIC_STUDENT"),
    ("register_student", {"event_id": "AI_STUDIO", "student_id": "DEMO001"}, "ALREADY_REGISTERED"),
    ("cancel_registration", {"event_id": "AI_STUDIO", "student_id": "DEMO004"}, "NOT_REGISTERED"),
    ("register_student", {"event_id": "AI_STUDIO"}, "ARGUMENT_CONTRACT"),
    ("send_email", {}, "UNKNOWN_TOOL"),
]
for name, arguments, expected_error in rejection_cases:
    before = validation_state.snapshot()
    result = validation_state.execute(name, arguments, approved=True)
    assert result.get("error") == expected_error and validation_state.snapshot() == before
    print("Expected rejection:", expected_error, "— registrations unchanged")
# Fill the second seat, then demonstrate capacity rejection and repeated cancellation.
assert validation_state.execute("register_student",
    {"event_id": "AI_STUDIO", "student_id": "DEMO002"}, approved=True)["ok"]
full_snapshot = validation_state.snapshot()
full_result = validation_state.execute("register_student",
    {"event_id": "AI_STUDIO", "student_id": "DEMO003"}, approved=True)
assert full_result["error"] == "EVENT_FULL" and validation_state.snapshot() == full_snapshot
print("Expected rejection: EVENT_FULL — registrations unchanged")
no_consent = validation_state.execute("cancel_registration",
    {"event_id": "AI_STUDIO", "student_id": "DEMO002"})
assert no_consent["error"] == "CLIENT_CONFIRMATION_REQUIRED" and validation_state.snapshot() == full_snapshot
assert validation_state.execute("cancel_registration",
    {"event_id": "AI_STUDIO", "student_id": "DEMO002"}, approved=True)["ok"]
repeat = validation_state.execute("cancel_registration",
    {"event_id": "AI_STUDIO", "student_id": "DEMO002"}, approved=True)
assert repeat["error"] == "NOT_REGISTERED"
# Check all configured events while keeping the one expected fixture registration explicit.
expected_registrations = {event: [] for event in EVENTS}
expected_registrations["AI_STUDIO"] = ["DEMO001"]
assert validation_state.snapshot() == expected_registrations
print("PASS: capacity, identity, duplicate, cancellation and client approval checks.")


## 7. Optional live model experiment

Add your own `GEMINI_API_KEY` in Colab Secrets, enable notebook access, set
`USE_LIVE_MODEL = True`, and rerun the config cell and this cell. Only this opted-in
path imports or installs `google-genai==2.12.0`; active Colab `google-auth`
requirements are preserved. The instructor must verify
`gemini-3.5-flash-lite` and account access before class.

Read each displayed simulated write. Type `CONFIRM` to approve that exact write,
or press Enter to decline. Never use real names or student IDs. Requests have a
60-second timeout and at most one automatic transient retry; quota/auth errors
do not retry. If a response stays pending, rerun this cell to resume the same turn
with stored tool results and counters. It will not ask to approve a completed write
a second time. `live_agent.reset()` intentionally resets the simulation and keeps
the consumed request budget. Live model prose is not proof of a real registration.


In [ ]:
# Live execution is opt-in; baseline fixture results never become live model answers.
LIVE_QUESTION = 'Check AI_STUDIO, register DEMO002, then check availability.'
live_reply = None
if not USE_LIVE_MODEL:
    print("Live model disabled. Baseline Run all requires no key, SDK install or network.")
else:
    # Rerunning this cell retains the existing simulation, history and request budget.
    existing = globals().get("live_agent")
    if existing is None:
        live_agent = ToolAgent(EventState(), GeminiSource(), EVENT_SCHEMAS, EVENT_SYSTEM,
                               approve_write=live_client_approval)
    else:
        live_agent = existing
        live_agent.__class__ = ToolAgent
        live_agent.state.__class__ = EventState
        print("Existing live state, trace, pending turn and request counter retained.")
    if live_agent.pending is not None:
        if live_agent.pending["question"] == LIVE_QUESTION and live_agent.can_resume:
            live_reply = show_attempt("Resume live response", live_agent.resume)
        else:
            print("PENDING: inspect the earlier turn and limits before another question.")
    elif live_agent.last_result is None:
        live_reply = show_attempt("Live response", lambda: live_agent.run(LIVE_QUESTION))
    else:
        print("This live example already completed; inspect its trace before a new request.")
    live_agent.show_trace()
    print("Physical requests used:", live_agent.requests_used, "Pending:", live_agent.pending is not None)


## 8. Student changes — not implemented in the supplied baseline

Make all three changes in your own copy while keeping the baseline checks passing.

1. **New event:** add `ROBOTICS_LAB` with capacity 1 to both synthetic state and
   discoverable event results. Register DEMO003 with client approval, then prove
   DEMO004 receives `EVENT_FULL` and no second seat appears.
2. **Personal schedule tool:** implement and declare read-only
   `get_student_schedule(student_id)` using known fake IDs. Reject an unknown ID.
   After registering DEMO002 in Python Jam, its schedule must contain exactly that
   event; after an approved cancellation it must be empty.
3. **Waitlist tool:** implement and declare `join_waitlist(event_id, student_id)`
   for a full event. Require client approval; reject an unknown ID, an already
   registered student, a duplicate waitlist entry and an event with free seats.
   A waitlist entry must not consume a registration seat or auto-enrol anyone.

**Checkpoints:** show separate model/fixture tool proposals, client decisions,
tool observations and before/after registrations. Explain the difference between
a proposed write, client approval, and a successful Python mutation. Use synthetic
data throughout; this project must never connect to a real college registration API.


### Your extension workspace
This unfinished cell is intentionally safe to run. Add your implementation and evidence here in your own copy.


In [ ]:
# TODO 1: add ROBOTICS_LAB and prove its one-seat limit through tool calls.
# TODO 2: implement get_student_schedule with its Python contract and declaration.
# TODO 3: add a separate approved waitlist that never changes seat counts.
print("Student extension workspace: 3 changes still to implement.")


## 9. Baseline receipt
This receipt covers the supplied synthetic baseline, including the declined write. It does not claim the student extensions are finished.


In [ ]:
# This receipt describes the completed supplied baseline, not unfinished student work.
baseline_evidence = {"mode": event_agent.source.mode, "synthetic": True,
    "baseline_complete": event_agent.last_result is not None and event_agent.pending is None,
    "tool_calls": len(event_agent.trace), "decision_requests": event_agent.requests_used,
    "state": event_state.snapshot(), "student_extensions_complete": False}
print(json.dumps(baseline_evidence, indent=2))
print("Keep your own extension trace and before/after state as submission evidence.")


## Source and provenance

Independently written classroom adaptation of the tool declaration → model choice →
tool result pattern demonstrated in the official
[OpenAI Agents Python basic tools example](https://github.com/openai/openai-agents-python/blob/main/examples/basic/tools.py)
and [repository README](https://github.com/openai/openai-agents-python/blob/main/README.md).
Those sources were read on 4 November 2026; upstream code was not executed or copied.
This notebook uses manual dispatch and optional Google Gen AI, not the OpenAI SDK.
No IBM, client, real student, bank or college operational data is included.
